### 02 — Spatiotemporal Analysis

This notebook aggregates cleaned AIS observations by **GFW spatial cell × month × year** and establishes the temporal/spatial summaries needed before regional trend analysis.

### Important preprocessing note

The source dataset used for this analysis was previously filtered to retain observations with fishing activity. Therefore, the current dataset represents fishing-active observations rather than all vessel presence/effort observations.

Consequently, `fishing_hours` is the primary measure of fishing effort. `fishing_ratio` is supplementary and should not be interpreted as the proportion of all vessel time spent fishing.

### 1. Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### 2. Load cleaned dataset

In [ ]:
DATA_FILE = Path("../outputs/processed/ais_cleaned.parquet")
ais = pd.read_parquet(DATA_FILE)
print("Shape:", ais.shape)
print("Date range:", ais.date.min(), "→", ais.date.max())
print("Unique vessels:", ais.mmsi.nunique())
print("Unique spatial cells:", ais[["cell_ll_lat","cell_ll_lon"]].drop_duplicates().shape[0])
print("Latitude:", ais.cell_ll_lat.min(), "→", ais.cell_ll_lat.max())
print("Longitude:", ais.cell_ll_lon.min(), "→", ais.cell_ll_lon.max())

### 3. Aggregate by spatial cell × month × year

In [ ]:
monthly_cell = ais.groupby(["year","month","cell_ll_lat","cell_ll_lon"], as_index=False).agg(total_hours=("hours","sum"), fishing_hours=("fishing_hours","sum"), active_vessels=("mmsi","nunique"), active_days=("date","nunique"))
print("Aggregated shape:", monthly_cell.shape)
display(monthly_cell.head())

### 4. Supplementary fishing ratio

In [ ]:
monthly_cell["fishing_ratio"] = np.where(monthly_cell.total_hours > 0, monthly_cell.fishing_hours/monthly_cell.total_hours, np.nan)
display(monthly_cell.fishing_ratio.describe())
ratio_one = int((monthly_cell.fishing_ratio == 1).sum())
print("Ratio exactly 1:", ratio_one)
print("Percentage:", round(100*ratio_one/len(monthly_cell),2), "%")
print("Groups where fishing_hours > total_hours:", int((monthly_cell.fishing_hours > monthly_cell.total_hours).sum()))

### 5. Overall yearly fishing effort

In [ ]:
yearly_summary = monthly_cell.groupby("year", as_index=False).agg(fishing_hours=("fishing_hours","sum"), total_hours=("total_hours","sum"), active_vessels=("active_vessels","sum"), active_days=("active_days","sum"))
display(yearly_summary)

### 6. Monthly fishing effort

In [ ]:
monthly_summary = monthly_cell.groupby("month", as_index=False).agg(fishing_hours=("fishing_hours","sum"), total_hours=("total_hours","sum"))
display(monthly_summary)

### 7. Year × month fishing-effort matrix

In [ ]:
year_month_effort = monthly_cell.groupby(["year","month"])["fishing_hours"].sum().unstack(fill_value=np.nan)
display(year_month_effort)

### 8. Annual effort plot

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(yearly_summary.year, yearly_summary.fishing_hours, marker="o")
plt.xlabel("Year"); plt.ylabel("Fishing hours"); plt.title("Annual Fishing Effort"); plt.grid(alpha=0.25); plt.show()

### 9. Monthly seasonality plot

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly_summary.month, monthly_summary.fishing_hours, marker="o")
plt.xticks(range(1,13)); plt.xlabel("Month"); plt.ylabel("Fishing hours"); plt.title("Fishing Effort by Month"); plt.grid(alpha=0.25); plt.show()

### 10. Spatial summary

In [ ]:
spatial_summary = monthly_cell.groupby(["cell_ll_lat","cell_ll_lon"], as_index=False).agg(total_fishing_hours=("fishing_hours","sum"), total_hours=("total_hours","sum"), active_vessels=("active_vessels","sum"), active_months=("month","count"))
display(spatial_summary.sort_values("total_fishing_hours", ascending=False).head(20))

### 11. Spatial fishing-effort map

In [ ]:
plt.figure(figsize=(11,7))
plt.scatter(spatial_summary.cell_ll_lon, spatial_summary.cell_ll_lat, c=spatial_summary.total_fishing_hours, s=12)
plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.title("Total Fishing Effort by GFW Spatial Cell")
plt.colorbar(label="Fishing hours"); plt.show()

### 12. Highest-effort cells

In [ ]:
top_cells = spatial_summary.sort_values("total_fishing_hours", ascending=False).head(25).reset_index(drop=True)
display(top_cells)

### 13. Year × spatial cell effort

In [ ]:
yearly_cell = monthly_cell.groupby(["year","cell_ll_lat","cell_ll_lon"], as_index=False).agg(fishing_hours=("fishing_hours","sum"), total_hours=("total_hours","sum"), active_vessels=("active_vessels","sum"), active_days=("active_days","sum"))
print("Year × cell rows:", len(yearly_cell))
display(yearly_cell.head())

### 14. Export analysis datasets

In [ ]:
ANALYSIS_DIR = Path("../outputs/analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
monthly_cell.to_parquet(ANALYSIS_DIR/"monthly_cell.parquet", index=False)
yearly_cell.to_parquet(ANALYSIS_DIR/"yearly_cell.parquet", index=False)
spatial_summary.to_parquet(ANALYSIS_DIR/"spatial_summary.parquet", index=False)
yearly_summary.to_csv(ANALYSIS_DIR/"yearly_summary.csv", index=False)
monthly_summary.to_csv(ANALYSIS_DIR/"monthly_summary.csv", index=False)
print("Analysis datasets saved to:", ANALYSIS_DIR.resolve())

### 15. Interpretation boundary

This notebook measures **AIS-derived fishing effort**, not fish biomass, catch, or true biological productivity. A decrease in `fishing_hours` therefore means a decrease in observed fishing effort; it does **not by itself prove** that fish productivity declined.

The next notebook should test whether fishing effort shows statistically meaningful regional trends and identify cells that are increasing, decreasing, or stable.